# Data Cleaning and Feature Engineering


In [ ]:
import os
project_root = os.path.abspath('..')
os.chdir(project_root)

In [ ]:
import pandas as pd
import numpy as np
import pickle
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
from sklearn.preprocessing import MinMaxScaler

load_dotenv()

db_user = os.getenv("DB_USER")
db_password = os.getenv("DB_PASSWORD")
db_host = os.getenv("DB_HOST")
db_port = os.getenv("DB_PORT")
db_name = os.getenv("DB_NAME")

engine = create_engine(
    f"mysql+pymysql://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}"
)

with engine.connect() as conn:
    conn.execute(text("SELECT 1"))
print("Database connection verified")

## 1. Load Market Prices
Wholesale cereal prices for Maize, Millet, and Sorghum across five markets — Tamale, Bolga, and Wa as the primary northern targets; Kumasi and Techiman as southern reference signals whose prices lead northern markets by 2–3 weeks.

In [ ]:
query = """
    SELECT
        p.date,
        p.market,
        p.market_id,
        p.commodity,
        p.price,
        p.pricetype,
        m.latitude,
        m.longitude,
        m.admin1
    FROM wfp_prices p
    JOIN wfp_markets m ON p.market_id = m.market_id
    WHERE p.commodity IN ('Maize', 'Millet', 'Sorghum')
    AND p.pricetype = 'Wholesale'
    AND p.market IN ('Tamale', 'Bolga', 'Wa', 'Kumasi', 'Techiman')
    ORDER BY p.market, p.commodity, p.date
"""

df = pd.read_sql(query, engine)
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month

print(f"total records loaded: {len(df)}")
print(f"markets: {sorted(df['market'].unique())}")
print(f"commodities: {sorted(df['commodity'].unique())}")
print(f"date range: {df['date'].min().date()} to {df['date'].max().date()}")
print(f"missing values: {df.isnull().sum().sum()}")
print()
print("records per market per commodity:")
print(df.groupby(['market', 'commodity']).size().unstack(fill_value=0))

## 2. Exchange Rate
Annual GHS/USD rate from FAOSTAT merged by year. Gives the LSTM a macro signal to separate inflation-driven price spikes (2022–2023 cedi collapse) from genuine seasonal supply-demand patterns.

In [ ]:
fx_query = """
    SELECT year, value as exchange_rate
    FROM ghana_exchange_rates
    WHERE months = 'Annual value'
    AND element = 'Local currency units per USD'
    ORDER BY year
"""

fx = pd.read_sql(fx_query, engine)
fx['year'] = fx['year'].astype(int)

# 2006-2007 annual values are in old Ghana cedis — divide by 10,000 to convert
fx.loc[fx['exchange_rate'] >= 100, 'exchange_rate'] = (
    fx.loc[fx['exchange_rate'] >= 100, 'exchange_rate'] / 10_000
)

fx = fx.groupby('year')['exchange_rate'].mean().reset_index()

print(f"exchange rate records after cleaning: {len(fx)}")
print(f"year range: {fx['year'].min()} to {fx['year'].max()}")
print()
print("exchange rates from 2006 onwards:")
print(fx[fx['year'] >= 2006].to_string(index=False))

df = df.merge(fx, on='year', how='left')
df = df.drop_duplicates(subset=['date', 'market', 'commodity']).reset_index(drop=True)

print(f"\nrows after exchange rate merge: {len(df)}")
print(f"missing exchange rates: {df['exchange_rate'].isnull().sum()}")

## 3. Producer Price Index
Farm gate price index (2014–2016 = 100) from FAOSTAT merged by year and commodity. Captures the distress-selling spread: when farm gate prices sit far below market prices, distress selling is active and the subsequent recovery tends to be stronger.

In [ ]:
# FAOSTAT has no maize producer prices for Ghana — millet/sorghum average used as proxy
pp_query = """
    SELECT year, item, value as producer_price_index
    FROM fao_producer_prices
    WHERE item IN ('Maize', 'Millet', 'Sorghum')
    AND months = 'Annual value'
    AND element = 'Producer Price Index (2014-2016 = 100)'
    ORDER BY item, year
"""

pp = pd.read_sql(pp_query, engine)
pp['year'] = pp['year'].astype(int)
pp = pp[pp['producer_price_index'] < 10000].copy()
pp = pp.groupby(['year', 'item'])['producer_price_index'].mean().reset_index()

pp_wide = pp.pivot(index='year', columns='item', values='producer_price_index').reset_index()
pp_wide.columns.name = None

# skipna=False requires both Millet and Sorghum to exist before computing the proxy
maize_proxy = pp_wide[['Millet', 'Sorghum']].mean(axis=1, skipna=False)
if 'Maize' in pp_wide.columns:
    pp_wide['Maize'] = pp_wide['Maize'].fillna(maize_proxy)
else:
    pp_wide['Maize'] = maize_proxy

print('producer price table from 2006 onwards:')
print(pp_wide[pp_wide['year'] >= 2006].to_string(index=False))

pp_long = pp_wide.melt(id_vars='year', var_name='commodity', value_name='producer_price_index')
df['year'] = df['date'].dt.year
df = df.merge(pp_long, on=['year', 'commodity'], how='left').drop(columns='year')

print(f'\nrows after producer price merge: {len(df)}')
print(f"missing producer index: {df['producer_price_index'].isnull().sum()}")
print()
print(df[['date', 'market', 'commodity', 'price', 'producer_price_index']].head(8))

## 4. Outlier Removal
IQR filter applied independently per market-commodity group to remove data entry errors while preserving genuine price extremes driven by the 2022–2023 currency shock.

In [ ]:
# IQR multiplier of 5 preserves genuine 2022-2023 cedi-collapse prices that multiplier 3 removed
n_before_outlier = len(df)
print(f"records before outlier removal: {n_before_outlier}")

def iqr_mask(series):
    q1, q3 = series.quantile(0.25), series.quantile(0.75)
    iqr = q3 - q1
    return series.between(q1 - 5 * iqr, q3 + 5 * iqr)

mask = df.groupby(["market", "commodity"])["price"].transform(iqr_mask)
df = df.loc[mask].reset_index(drop=True)

print(f"records after outlier removal: {len(df)}")
print(f"records removed: {n_before_outlier - len(df)}")
print()
print("records per market per commodity after cleaning:")
print(df.groupby(["market", "commodity"]).size().unstack(fill_value=0))
print()
print(f"2023 records kept: {len(df[df['date'].dt.year == 2023])}")
print(f"date range: {df['date'].min().date()} to {df['date'].max().date()}")

## 5. Feature Engineering
Price scaled per market-commodity pair so the LSTM learns seasonal patterns, not price-level differences between markets. Exchange rate and producer price index scaled globally — they are national indicators, not market-specific. Month encoded as sin/cos so the model treats December and January as adjacent.

In [ ]:
scalers = {}
df['price_scaled'] = 0.0

for (market, commodity), group in df.groupby(['market', 'commodity']):
    scaler = MinMaxScaler()
    idx = group.index
    df.loc[idx, 'price_scaled'] = scaler.fit_transform(df.loc[idx, ['price']]).flatten()
    scalers[f"{commodity}_{market}"] = scaler

df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)

print(f"scalers created: {len(scalers)}")
print(f"scaler keys: {list(scalers.keys())}")
print()
print(f"price_scaled range: {df['price_scaled'].min():.2f} to {df['price_scaled'].max():.2f}")
print(f"month_sin range: {df['month_sin'].min():.2f} to {df['month_sin'].max():.2f}")
print(f"month_cos range: {df['month_cos'].min():.2f} to {df['month_cos'].max():.2f}")
print()
print(df[['date', 'market', 'commodity', 'price', 'price_scaled', 'month_sin', 'month_cos']].head(6))

In [ ]:
fx_scaler = MinMaxScaler()
df['fx_scaled'] = fx_scaler.fit_transform(df[['exchange_rate']]).flatten()
scalers['exchange_rate'] = fx_scaler

pp_scaler = MinMaxScaler()
df['pp_scaled'] = pp_scaler.fit_transform(df[['producer_price_index']]).flatten()
scalers['producer_price_index'] = pp_scaler

print(f"total scalers: {len(scalers)}")
print(f"fx_scaled range: {df['fx_scaled'].min():.2f} to {df['fx_scaled'].max():.2f}")
print(f"pp_scaled range: {df['pp_scaled'].min():.2f} to {df['pp_scaled'].max():.2f}")
print()
print(df[['date', 'market', 'commodity', 'price_scaled', 'month_sin', 'month_cos', 'fx_scaled', 'pp_scaled']].head(6))

## 6. Save Outputs
Scalers must be saved alongside the model — they are required at inference time to convert normalised LSTM output back to Ghana cedis before presenting to the farmer.

In [ ]:
import os

os.makedirs('app/ml/models', exist_ok=True)

with open('app/ml/models/price_scalers.pkl', 'wb') as f:
    pickle.dump(scalers, f)

print(f"saved {len(scalers)} scalers to app/ml/models/price_scalers.pkl")

final_cols = [
    'date', 'market', 'commodity', 'price',
    'latitude', 'longitude',
    'price_scaled', 'month_sin', 'month_cos',
    'fx_scaled', 'pp_scaled'
]

df_clean = df[final_cols].copy()

n_before_drop = len(df_clean)
df_clean = df_clean.dropna().reset_index(drop=True)
print(f"rows dropped by dropna (missing fx or producer index): {n_before_drop - len(df_clean)}")

os.makedirs('data/processed', exist_ok=True)
df_clean.to_csv('data/processed/cereals_merged_clean.csv', index=False)

print()
print(f"total rows: {len(df_clean)}")
print(f"columns: {list(df_clean.columns)}")
print(f"markets: {sorted(df_clean['market'].unique())}")
print(f"commodities: {sorted(df_clean['commodity'].unique())}")
print(f"date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"missing values: {df_clean.isnull().sum().sum()}")
print()
print("records per market per commodity:")
print(df_clean.groupby(['market', 'commodity']).size().unstack(fill_value=0))
print()
print(f"scalers file exists: {os.path.exists('app/ml/models/price_scalers.pkl')}")
print(f"clean csv exists: {os.path.exists('data/processed/cereals_merged_clean.csv')}")